In [1]:
"""
EDA 2 - Pricing A/B Test (Delivery Fee)
Goal: find the delivery-fee set-up that keeps revenue the same (or better) while driving more orders.
Revenue = order_value * 20% commission + delivery_fee
Run:  python pricing_ab_test_analysis.py   (needs pandas, numpy, scipy, matplotlib, openpyxl)
"""
import os, json, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

import glob
DATA = glob.glob("*.xlsx")[0]   # picks the Excel file in this folder, whatever its name
OUT = "charts/"                 # charts are saved here (folder is created automatically)
os.makedirs("charts", exist_ok=True)
R = {}                                    # results dictionary (saved to results.json)
C_COL, T_COL = "#8A8F98", "#E8590C"       # control = grey, test = orange
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# ------------------------------------------------------------------ 1. Load & clean
df = pd.read_excel(DATA)
df["variant"] = df["variant_transaction"]
df["period"] = np.where(df.hour < 12, "Morning (<12)", np.where(df.hour >= 18, "Evening (>=18)", "Afternoon (12-17)"))
df["has_fee"] = np.where(df.delivery_fee > 0, "With fee", "Free delivery")
assert df.isna().sum().sum() == 0 and df.code.is_unique
assert np.allclose(df.order_value * 0.2 + df.delivery_fee, df.revenue)   # revenue formula verified
ctl, tst = df[df.variant == "Control"], df[df.variant == "Test"]
R["rows"] = len(df); R["n_ctl"] = len(ctl); R["n_tst"] = len(tst)

def ci_mean_diff(a, b):
    d = b.mean() - a.mean(); se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    return d, d - 1.96*se, d + 1.96*se

def compare(col, data=df):
    a, b = data[data.variant == "Control"][col], data[data.variant == "Test"][col]
    t, p = stats.ttest_ind(b, a, equal_var=False)
    u, pu = stats.mannwhitneyu(b, a, alternative="two-sided")
    d, lo, hi = ci_mean_diff(a, b)
    return dict(ctl_mean=a.mean(), tst_mean=b.mean(), diff=d, pct=d/a.mean()*100, ci_lo=lo, ci_hi=hi,
                t=t, p_welch=p, p_mwu=pu, ctl_sd=a.std(), tst_sd=b.std())

# ------------------------------------------------------------------ 2. KPI table
def kpis(x):
    return dict(orders=len(x), total_order_value=x.order_value.sum(), aov=x.order_value.mean(),
                avg_fee=x.delivery_fee.mean(), total_fee=x.delivery_fee.sum(),
                revenue=x.revenue.sum(), rev_per_order=x.revenue.mean(),
                commission=(x.order_value*0.2).sum())
R["kpi"] = {"Control": kpis(ctl), "Test": kpis(tst)}
days = df.date.nunique(); R["days"] = days

# ------------------------------------------------------------------ 3. EDA
# Q1 order value by hour
hr = df.groupby("hour").order_value.agg(["mean", "median", "std", "count"])
R["hour_ov"] = hr.round(2).reset_index().to_dict("records")
kw = stats.kruskal(*[g.order_value.values for _, g in df.groupby("hour")])
R["kruskal_hour_ov"] = [kw.statistic, kw.pvalue]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
df.boxplot(column="order_value", by="hour", ax=ax[0], showfliers=False, grid=False); ax[0].set_title("Order value by hour (outliers hidden)"); ax[0].set_xlabel("Hour"); ax[0].set_ylabel("Order value"); fig.suptitle("")
ax[1].bar(hr.index, hr["mean"], color="#1F6F8B"); ax[1].axhline(df.order_value.mean(), color="k", ls="--", lw=1, label=f"Overall mean {df.order_value.mean():.1f}")
ax[1].set_title("Average order value by hour"); ax[1].set_xlabel("Hour"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT + "q1_hour_ov.png", dpi=150); plt.close()

# Q2 revenue morning vs evening
m, e = df[df.hour < 12].revenue, df[df.hour >= 18].revenue
R["morning_evening"] = dict(m_mean=m.mean(), m_med=m.median(), m_n=len(m), e_mean=e.mean(), e_med=e.median(), e_n=len(e),
                            p_mwu=stats.mannwhitneyu(m, e).pvalue, p_welch=stats.ttest_ind(m, e, equal_var=False).pvalue)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(m, bins=50, alpha=.65, density=True, range=(0, 30), label="Morning (<12)", color="#1F6F8B")
ax.hist(e, bins=50, alpha=.65, density=True, range=(0, 30), label="Evening (>=18)", color="#E8590C")
ax.set_title("Revenue per order: morning vs evening"); ax.set_xlabel("Revenue"); ax.legend()
plt.tight_layout(); plt.savefig(OUT + "q2_morning_evening.png", dpi=150); plt.close()

# Q3 delivery fee vs revenue
R["corr_fee_rev"] = dict(pearson=df.delivery_fee.corr(df.revenue), spearman=df.delivery_fee.corr(df.revenue, method="spearman"))
fee_tbl = df.groupby(["variant", "delivery_fee"]).agg(orders=("code", "count"), aov=("order_value", "mean"), rev=("revenue", "mean")).round(2).reset_index()
R["fee_tbl"] = fee_tbl.to_dict("records")
fig, ax = plt.subplots(figsize=(8, 4.5))
g = df.groupby("delivery_fee").agg(rev=("revenue", "mean"), n=("code", "count"))
ax.bar(g.index.astype(str), g.rev, color="#1F6F8B"); ax.set_title("Average revenue per order by delivery fee"); ax.set_xlabel("Delivery fee"); ax.set_ylabel("Avg revenue")
for i, (r_, n_) in enumerate(zip(g.rev, g.n)): ax.text(i, r_ + .15, f"n={n_}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(OUT + "q3_fee_revenue.png", dpi=150); plt.close()

# ------------------------------------------------------------------ 4. A/B tests
R["ab"] = {c: compare(c) for c in ["revenue", "order_value", "delivery_fee"]}
# order_value outliers: log-scale test + bootstrap
a, b = np.log(ctl.order_value), np.log(tst.order_value)
R["ab_log_ov"] = dict(p=stats.ttest_ind(b, a, equal_var=False).pvalue, ctl=np.exp(a.mean()), tst=np.exp(b.mean()))
rng = np.random.default_rng(42)
boot = [rng.choice(tst.revenue.values, len(tst)).mean() - rng.choice(ctl.revenue.values, len(ctl)).mean() for _ in range(3000)]
R["boot_rev_diff"] = [np.percentile(boot, 2.5), np.percentile(boot, 97.5)]

# Orders: daily comparison (same 31 days, both arms)
daily = df.groupby(["date", "variant"]).agg(orders=("code", "count"), revenue=("revenue", "sum")).unstack()
t_o = stats.ttest_rel(daily["orders"]["Test"], daily["orders"]["Control"]); t_r = stats.ttest_rel(daily["revenue"]["Test"], daily["revenue"]["Control"])
R["daily"] = dict(ord_ctl=daily["orders"]["Control"].mean(), ord_tst=daily["orders"]["Test"].mean(), p_orders=t_o.pvalue,
                  rev_ctl=daily["revenue"]["Control"].mean(), rev_tst=daily["revenue"]["Test"].mean(), p_rev=t_r.pvalue,
                  days_test_more_orders=int((daily["orders"]["Test"] > daily["orders"]["Control"]).sum()),
                  days_test_more_rev=int((daily["revenue"]["Test"] > daily["revenue"]["Control"]).sum()))
# order count chi-square vs 50/50 split (can't verify true traffic split -> flag as assumption)
R["split_chi"] = stats.chisquare([len(ctl), len(tst)]).pvalue

# Revenue by variant chart
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, (col, ttl) in enumerate([("orders", "Orders"), ("revenue", "Revenue (total)"), ("aov", "Avg order value")]):
    v = [R["kpi"]["Control"][{"orders": "orders", "revenue": "revenue", "aov": "aov"}[col]], R["kpi"]["Test"][{"orders": "orders", "revenue": "revenue", "aov": "aov"}[col]]]
    ax[i].bar(["Control", "Test"], v, color=[C_COL, T_COL]); ax[i].set_title(ttl)
    for j, x in enumerate(v): ax[i].text(j, x*1.01, f"{x:,.0f}" if col != "aov" else f"{x:.2f}", ha="center")
plt.tight_layout(); plt.savefig(OUT + "ab_kpis.png", dpi=150); plt.close()

# Hour-wise treatment effect
hh = df.groupby(["hour", "variant"]).agg(orders=("code", "count"), rev_total=("revenue", "sum"), rev_avg=("revenue", "mean"), aov=("order_value", "mean")).unstack()
rows = []
for h, g in df.groupby("hour"):
    a_, b_ = g[g.variant == "Control"].revenue, g[g.variant == "Test"].revenue
    p = stats.mannwhitneyu(b_, a_).pvalue; pw = stats.ttest_ind(b_, a_, equal_var=False).pvalue
    rows.append(dict(hour=h, ctl_orders=len(a_), tst_orders=len(b_), orders_lift_pct=(len(b_)/len(a_)-1)*100,
                     ctl_rev_avg=a_.mean(), tst_rev_avg=b_.mean(), rev_avg_lift_pct=(b_.mean()/a_.mean()-1)*100,
                     ctl_rev_total=a_.sum(), tst_rev_total=b_.sum(), p_welch=pw, p_mwu=p,
                     ctl_aov=g[g.variant == "Control"].order_value.mean(), tst_aov=g[g.variant == "Test"].order_value.mean()))
hour_tbl = pd.DataFrame(rows)
# Bonferroni across 14 hours
hour_tbl["sig_bonf"] = hour_tbl.p_welch < 0.05/len(hour_tbl)
R["hour_tbl"] = hour_tbl.round(3).to_dict("records")
R["hour_interaction"] = None
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
w = .4; x = hour_tbl.hour.values
ax[0].bar(x-w/2, hour_tbl.ctl_rev_avg, w, color=C_COL, label="Control"); ax[0].bar(x+w/2, hour_tbl.tst_rev_avg, w, color=T_COL, label="Test")
ax[0].set_title("Avg revenue per order by hour"); ax[0].set_xlabel("Hour"); ax[0].legend()
ax[1].bar(x-w/2, hour_tbl.ctl_orders, w, color=C_COL, label="Control"); ax[1].bar(x+w/2, hour_tbl.tst_orders, w, color=T_COL, label="Test")
ax[1].set_title("Orders by hour"); ax[1].set_xlabel("Hour"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT + "ab_hour.png", dpi=150); plt.close()

# Variance of order value
lev = stats.levene(ctl.order_value, tst.order_value); lev_med = stats.levene(ctl.order_value, tst.order_value, center="median")
R["variance"] = dict(ctl_var=ctl.order_value.var(), tst_var=tst.order_value.var(), ctl_sd=ctl.order_value.std(), tst_sd=tst.order_value.std(),
                     ctl_cv=ctl.order_value.std()/ctl.order_value.mean(), tst_cv=tst.order_value.std()/tst.order_value.mean(),
                     levene_p=lev.pvalue, brown_forsythe_p=lev_med.pvalue,
                     ctl_iqr=ctl.order_value.quantile(.75)-ctl.order_value.quantile(.25), tst_iqr=tst.order_value.quantile(.75)-tst.order_value.quantile(.25),
                     ctl_p95=ctl.order_value.quantile(.95), tst_p95=tst.order_value.quantile(.95),
                     ctl_skew=ctl.order_value.skew(), tst_skew=tst.order_value.skew())
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].hist(ctl.order_value, bins=60, range=(0, 150), alpha=.6, color=C_COL, density=True, label="Control"); ax[0].hist(tst.order_value, bins=60, range=(0, 150), alpha=.6, color=T_COL, density=True, label="Test")
ax[0].set_title("Order value distribution"); ax[0].legend(); ax[0].set_xlabel("Order value")
ax[1].boxplot([ctl.order_value, tst.order_value], tick_labels=["Control", "Test"], showfliers=False, patch_artist=True, boxprops=dict(facecolor="#dde")); ax[1].set_title("Order value spread (outliers hidden)")
plt.tight_layout(); plt.savefig(OUT + "ab_variance.png", dpi=150); plt.close()

# ------------------------------------------------------------------ 5. Additional questions
# Fee presence
fee = df.groupby(["variant", "has_fee"]).agg(orders=("code", "count"), aov=("order_value", "mean"), rev_avg=("revenue", "mean"), rev_total=("revenue", "sum"), min_ov=("order_value", "min")).round(2).reset_index()
R["fee_presence"] = fee.to_dict("records")
free = tst[tst.delivery_fee == 0]; paid = tst[tst.delivery_fee > 0]
R["free_vs_paid"] = dict(rev_p=stats.mannwhitneyu(free.revenue, paid.revenue).pvalue, ov_p=stats.mannwhitneyu(free.order_value, paid.order_value).pvalue,
                         free_share=len(free)/len(tst), free_min=free.order_value.min(), paid_max_p99=paid.order_value.quantile(.99))
# Threshold effect: share of orders >= 33 and revenue by order-value band
bands = pd.cut(df.order_value, [0, 15, 25, 33, 50, 80, 1000], labels=["<15", "15-25", "25-33", "33-50", "50-80", "80+"])
bt = df.groupby([bands, "variant"], observed=True).agg(orders=("code", "count"), rev_avg=("revenue", "mean"), fee_avg=("delivery_fee", "mean")).round(2).reset_index()
bt["band"] = bt["order_value"].astype(str); bt = bt.drop(columns="order_value")
R["bands"] = bt.to_dict("records")
R["share_ge33"] = dict(ctl=(ctl.order_value >= 33).mean(), tst=(tst.order_value >= 33).mean())
ge = lambda d: d[d.order_value >= 33]; lt = lambda d: d[d.order_value < 33]
R["ge33"] = dict(ctl_n=len(ge(ctl)), tst_n=len(ge(tst)), ctl_rev=ge(ctl).revenue.sum(), tst_rev=ge(tst).revenue.sum(), ctl_rev_avg=ge(ctl).revenue.mean(), tst_rev_avg=ge(tst).revenue.mean(),
                 ctl_aov=ge(ctl).order_value.mean(), tst_aov=ge(tst).order_value.mean())
R["lt33"] = dict(ctl_n=len(lt(ctl)), tst_n=len(lt(tst)), ctl_rev=lt(ctl).revenue.sum(), tst_rev=lt(tst).revenue.sum(), ctl_rev_avg=lt(ctl).revenue.mean(), tst_rev_avg=lt(tst).revenue.mean(),
                 ctl_aov=lt(ctl).order_value.mean(), tst_aov=lt(tst).order_value.mean(), ctl_fee=lt(ctl).delivery_fee.mean(), tst_fee=lt(tst).delivery_fee.mean())

# Correlation order value vs revenue
R["corr_ov_rev"] = {v: dict(pearson=g.order_value.corr(g.revenue), spearman=g.order_value.corr(g.revenue, method="spearman")) for v, g in df.groupby("variant")}
R["corr_ov_rev"]["All"] = dict(pearson=df.order_value.corr(df.revenue))
sl = {v: stats.linregress(g.order_value, g.revenue) for v, g in df.groupby("variant")}
R["slope"] = {v: dict(slope=s.slope, intercept=s.intercept, r2=s.rvalue**2) for v, s in sl.items()}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for v, c in [("Control", C_COL), ("Test", T_COL)]:
    g = df[df.variant == v].sample(1500, random_state=1) if len(df[df.variant == v]) > 1500 else df[df.variant == v]
    ax.scatter(g.order_value, g.revenue, s=8, alpha=.35, color=c, label=v)
ax.set_xlim(0, 120); ax.set_ylim(0, 30); ax.set_xlabel("Order value"); ax.set_ylabel("Revenue"); ax.set_title("Order value vs revenue"); ax.legend()
plt.tight_layout(); plt.savefig(OUT + "add_corr.png", dpi=150); plt.close()

# Fee presence influence on order value within each group
R["fee_ov_ctl"] = "Control always charges 2.29, so fee presence cannot vary there"
R["test_fee_ov_corr"] = tst.delivery_fee.corr(tst.order_value, method="spearman")
# Fee vs order value chart (test)
fig, ax = plt.subplots(figsize=(8, 4.5))
g = tst.groupby("delivery_fee").order_value.mean()
ax.bar(g.index.astype(str), g.values, color=T_COL); ax.set_title("Test group: avg order value by delivery fee tier"); ax.set_xlabel("Delivery fee"); ax.set_ylabel("Avg order value")
plt.tight_layout(); plt.savefig(OUT + "add_fee_ov.png", dpi=150); plt.close()

# Revenue by hour per group is in hour_tbl. Fee tier by hour in test
R["test_fee_by_hour"] = tst.groupby("hour").delivery_fee.mean().round(2).to_dict()

# Revenue-neutral view: revenue per order & orders; break-even
R["lift"] = dict(orders_pct=(len(tst)/len(ctl)-1)*100, rev_total_pct=(tst.revenue.sum()/ctl.revenue.sum()-1)*100,
                 rev_per_order_pct=(tst.revenue.mean()/ctl.revenue.mean()-1)*100, aov_pct=(tst.order_value.mean()/ctl.order_value.mean()-1)*100)
# Post-hoc: if the arms were exposed equally, orders per arm-day
R["orders_per_day"] = dict(ctl=len(ctl)/days, tst=len(tst)/days)

# ------------------------------------------------------------------ 6. Save
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return float(o)
    if isinstance(o, (np.integer, int)): return int(o)
    if isinstance(o, (np.bool_, bool)): return bool(o)
    return o
json.dump(clean(R), open("results.json", "w"), indent=1)
hour_tbl.round(3).to_csv("hour_table.csv", index=False)
print("Done. Charts in ./charts, numbers in results.json and hour_table.csv")



/opt/anaconda3/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Done. Charts in ./charts, numbers in results.json and hour_table.csv
